# Heart Disease Classification — PyTorch Experiment Framework

**Dataset:** Heart Disease UCI (`ineubytes/heart-disease-dataset`) — 1025 samples, 13 features, binary target  
**Task:** Binary classification — predict presence of heart disease  
**Design:** Two-phase ablation study
- **Phase 1:** All regularizers × fixed optimizer → identify best regularizer
- **Phase 2:** All optimizers × best regularizer from Phase 1 → identify best optimizer
- **Final:** Retrain best config with Stochastic Weight Averaging (SWA)

All experiment lists are defined once in the **Configuration** cell.  
Adding or removing entries automatically propagates through the entire pipeline.

## 0. Setup & Imports

In [ ]:
import os, warnings, random
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Configuration

> **Single source of truth.** Modify `REGULARIZERS` or `OPTIMIZERS` here - every downstream cell adapts automatically.

In [ ]:
# ============================================================
#  REGULARIZER LIST
#  Each dict must have 'name' (display label) and 'type'
#  Supported types: none | l1 | l2 | dropout | label_smooth | bn_dropout
# ============================================================
REGULARIZERS = [
    {'name': 'Baseline',          'type': 'none'},
    {'name': 'L1',                'type': 'l1',           'l1_lambda': 1e-4},
    {'name': 'L2 (WeightDecay)',  'type': 'l2',           'weight_decay': 1e-4},
    {'name': 'Dropout',           'type': 'dropout',      'rate': 0.4},
    {'name': 'LabelSmoothing',    'type': 'label_smooth', 'smoothing': 0.1},
    {'name': 'BatchNorm+Dropout', 'type': 'bn_dropout',   'rate': 0.3},
]

# ============================================================
#  OPTIMIZER LIST
#  Each dict must have 'name' (display label) and 'type'
#  Supported types: sgd | adam | adamw | rmsprop
# ============================================================
OPTIMIZERS = [
    {'name': 'SGD',          'type': 'sgd',     'lr': 0.01},
    {'name': 'SGD+Momentum', 'type': 'sgd',     'lr': 0.01, 'momentum': 0.9},
    {'name': 'Adam',         'type': 'adam',    'lr': 1e-3},
    {'name': 'AdamW',        'type': 'adamw',   'lr': 1e-3, 'weight_decay': 1e-4},
    {'name': 'RMSprop',      'type': 'rmsprop', 'lr': 1e-3},
]

# Phase 1 uses this fixed optimizer to compare all regularizers
PHASE1_OPTIMIZER_NAME = 'Adam'

# ============================================================
#  TRAINING HYPERPARAMETERS
# ============================================================
EPOCHS         = 150
BATCH_SIZE     = 32
EARLY_STOP_PAT = 20      # stop if val_loss does not improve for N epochs
SWA_START_FRAC = 0.75    # SWA averaging begins at this fraction of EPOCHS

# MODEL ARCHITECTURE - INPUT_DIM is derived from data after loading
HIDDEN1 = 64
HIDDEN2 = 32

# SHARED METRIC MAP - used in all results visualisations
METRIC_MAP = {'test_roc_auc': 'ROC-AUC', 'test_f1': 'F1-Score', 'test_accuracy': 'Accuracy'}

# ============================================================
#  DATA LOADING MODE
#  False (default): download automatically via Kaggle API.
#  True           : skip Kaggle entirely and load data/heart.csv from disk.
#                   Use this if you have no Kaggle account, are offline, or
#                   placed the CSV manually after a failed API download.
# ============================================================
USE_LOCAL_CSV = False

print(f'Regularizers: {[r["name"] for r in REGULARIZERS]}')
print(f'Optimizers:   {[o["name"] for o in OPTIMIZERS]}')
print(f'Phase 1 optimizer: {PHASE1_OPTIMIZER_NAME}')
print(f'Total Phase-1 runs: {len(REGULARIZERS)}')
print(f'Total Phase-2 runs: {len(OPTIMIZERS)}')
print(f'USE_LOCAL_CSV: {USE_LOCAL_CSV}')

## 2. Data Loading (Kaggle API)

### One-time credential setup

The dataset is downloaded automatically via the Kaggle API. You need credentials once:

**Option A — OAuth (recommended)**
```powershell
.venv\Scripts\kaggle.exe auth login
```

**Option B — API token file**
1. Go to [kaggle.com/settings/api](https://www.kaggle.com/settings/api) → *Generate New Token*
2. Place `kaggle.json` at `~/.kaggle/kaggle.json`

**Option C — `KaggleToken.py` (local dev shortcut, git-ignored)**
```python
KAGGLE_API_TOKEN = "KGAT_xxxxxxxxxxxxxxxx"
```
The cell below reads it automatically if the file exists.

**Credential priority order:**  `KaggleToken.py` → `KAGGLE_API_TOKEN` env var → `kaggle.json` → `access_token`

In [ ]:
DATA_DIR     = Path('./data')
DATA_DIR.mkdir(exist_ok=True)
DATASET_SLUG = 'ineubytes/heart-disease-dataset'
CSV_FILE     = DATA_DIR / 'heart.csv'

if USE_LOCAL_CSV:
    # ── Local-only mode ───────────────────────────────────────────────
    if not CSV_FILE.exists():
        raise FileNotFoundError(
            f'\nUSE_LOCAL_CSV = True but file not found: {CSV_FILE}\n'
            'Download heart.csv manually from:\n'
            f'  https://www.kaggle.com/datasets/{DATASET_SLUG}\n'
            'and place it at:  data/heart.csv'
        )
    print(f'\u2713 USE_LOCAL_CSV = True — loading from {CSV_FILE}')

else:
    # ── Kaggle credential resolution — multi-method, priority order ───
    #   1. KaggleToken.py  (git-ignored local file)
    #   2. KAGGLE_API_TOKEN environment variable
    #   3. ~/.kaggle/kaggle.json
    #   4. ~/.kaggle/access_token
    _token_file = Path('KaggleToken.py')
    _kaggle_dir = Path.home() / '.kaggle'
    _cred_found = False

    if _token_file.exists():
        import importlib.util as _ilu
        _spec = _ilu.spec_from_file_location('KaggleToken', _token_file)
        _mod  = _ilu.module_from_spec(_spec)
        _spec.loader.exec_module(_mod)
        os.environ['KAGGLE_API_TOKEN'] = _mod.KAGGLE_API_TOKEN
        print('\u2713 Credentials loaded from KaggleToken.py')
        _cred_found = True
    elif os.environ.get('KAGGLE_API_TOKEN'):
        print('\u2713 Using KAGGLE_API_TOKEN environment variable')
        _cred_found = True
    elif (_kaggle_dir / 'kaggle.json').exists():
        print(f'\u2713 Using {_kaggle_dir / "kaggle.json"}')
        _cred_found = True
    elif (_kaggle_dir / 'access_token').exists():
        print(f'\u2713 Using {_kaggle_dir / "access_token"}')
        _cred_found = True

    if not _cred_found:
        if CSV_FILE.exists():
            print('\u26a0 No Kaggle credentials found — using existing local CSV as fallback.')
        else:
            raise EnvironmentError(
                '\nNo Kaggle credentials found and no local CSV available.\n'
                'Options:\n'
                '  A. Create KaggleToken.py:  KAGGLE_API_TOKEN = "KGAT_xxx"\n'
                '  B. Set env var:            KAGGLE_API_TOKEN=KGAT_xxx\n'
                '  C. Place kaggle.json at:   ~/.kaggle/kaggle.json\n'
                '  D. Manual download + set USE_LOCAL_CSV = True in the Configuration cell:\n'
                f'     Download from  https://www.kaggle.com/datasets/{DATASET_SLUG}\n'
                '     Place CSV at   data/heart.csv'
            )

    # ── Dataset download ──────────────────────────────────────────────
    if _cred_found and not CSV_FILE.exists():
        import kaggle
        try:
            print(f"\nDownloading '{DATASET_SLUG}' from Kaggle...")
            kaggle.api.authenticate()
            kaggle.api.dataset_download_files(DATASET_SLUG, path=str(DATA_DIR), unzip=True)
            for _f in DATA_DIR.glob('*.zip'):
                _f.unlink()
            print('Download complete.')
        except Exception as _exc:
            if CSV_FILE.exists():
                print(f'\u26a0 Kaggle download failed ({_exc}). Using existing local CSV as fallback.')
            else:
                raise EnvironmentError(
                    f'\nKaggle API error: {_exc}\n'
                    'Your token may be invalid or expired.\n'
                    'Fix options:\n'
                    '  - Regenerate token at https://www.kaggle.com/settings/api\n'
                    '  - Or: set USE_LOCAL_CSV = True and place heart.csv at data/heart.csv'
                )
    elif _cred_found:
        print(f'\nDataset already present at: {CSV_FILE}')

# ── Load & validate ───────────────────────────────────────────────
df_raw = pd.read_csv(CSV_FILE)

_expected_cols = {'age','sex','cp','trestbps','chol','fbs','restecg',
                  'thalach','exang','oldpeak','slope','ca','thal','target'}
_missing = _expected_cols - set(df_raw.columns)
assert not _missing, f'Unexpected CSV schema - missing columns: {_missing}'
assert set(df_raw['target'].unique()).issubset({0, 1}), 'target is not binary (0/1)'

print(f'\nShape : {df_raw.shape}')
print(f'Target: {df_raw["target"].value_counts().to_dict()}')
df_raw.head()

## 3. Exploratory Data Analysis

In [ ]:
# ── 1. Target distribution ────────────────────────────────────
target_counts = df_raw['target'].value_counts().reset_index()
target_counts.columns = ['target', 'count']
target_counts['label'] = target_counts['target'].map({0: 'No Disease', 1: 'Has Disease'})
target_counts['pct'] = (target_counts['count'] / target_counts['count'].sum() * 100).round(1)
target_counts['text'] = target_counts.apply(lambda r: f"{r['count']} ({r['pct']}%)", axis=1)

px.bar(
    target_counts, x='label', y='count', color='label',
    title='Target Distribution',
    color_discrete_sequence=px.colors.qualitative.Set2,
    text='text', template='plotly_white',
).update_traces(textposition='outside').update_layout(showlegend=False).show()

# ── 2. Feature–target correlation bar ─────────────────────────
# Most actionable EDA view for classification: which features drive the target?
feat_corr = (
    df_raw.corrwith(df_raw['target'])
    .drop('target')
    .sort_values(key=abs, ascending=True)
)
px.bar(
    x=feat_corr.values, y=feat_corr.index,
    orientation='h',
    title='Feature Correlation with Target (Pearson r)',
    color=feat_corr.values,
    color_continuous_scale='RdBu_r',
    color_continuous_midpoint=0,
    template='plotly_white',
    labels={'x': 'Pearson r with target', 'y': 'Feature'},
).update_layout(coloraxis_showscale=False, height=450).show()

# ── 3. Correlation heatmap (lower triangle only) ───────────────
corr = df_raw.corr(numeric_only=True)
corr_display = corr.mask(np.triu(np.ones(corr.shape, dtype=bool), k=1))
px.imshow(
    corr_display, text_auto='.2f',
    title='Feature Correlation Matrix (lower triangle)',
    color_continuous_scale='RdBu_r', zmin=-1, zmax=1, template='plotly_white',
    aspect='auto',
).show()

# ── 4. Box plots: class separation per feature ────────────────
# Box plots reveal median shift and IQR overlap between classes -
# the primary visual signal for each feature's predictive power.
numeric_cols = [c for c in df_raw.columns if c != 'target']
fig_box = make_subplots(rows=3, cols=5, subplot_titles=numeric_cols[:13])
_class_style = [(0, 'No Disease', '#636EFA'), (1, 'Has Disease', '#EF553B')]
for i, col in enumerate(numeric_cols[:13]):
    r_idx, c_idx = divmod(i, 5)
    for t_val, label, color in _class_style:
        fig_box.add_trace(
            go.Box(
                y=df_raw[df_raw['target'] == t_val][col],
                name=label, marker_color=color,
                showlegend=(i == 0), legendgroup=label,
                boxmean=True,
            ),
            row=r_idx + 1, col=c_idx + 1,
        )
fig_box.update_layout(
    title='Feature Distributions by Class (box = median/IQR, dot = mean)',
    height=650, template='plotly_white', boxmode='group',
)
fig_box.show()

## 4. Preprocessing

In [ ]:
FEATURE_COLS = [c for c in df_raw.columns if c != 'target']
TARGET_COL   = 'target'

assert df_raw[FEATURE_COLS].isnull().sum().sum() == 0, 'Missing values found in features!'

X = df_raw[FEATURE_COLS].values.astype(np.float32)
y = df_raw[TARGET_COL].values.astype(np.float32)

# Derive INPUT_DIM from data so it stays correct if the dataset changes
INPUT_DIM = X.shape[1]
print(f'Input features: {INPUT_DIM}  ({FEATURE_COLS})')

# Stratified 70 / 15 / 15 split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

def to_loader(X, y, shuffle=False):
    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32),
    )
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

train_loader = to_loader(X_train, y_train, shuffle=True)
val_loader   = to_loader(X_val,   y_val)
test_loader  = to_loader(X_test,  y_test)

print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')
print(f'Class balance - Train: {y_train.mean():.3f}  Val: {y_val.mean():.3f}  Test: {y_test.mean():.3f}')

## 5. Model Factory & Training Engine

### Architecture rationale
A shallow MLP (2 hidden layers) is appropriate for a 13-feature tabular dataset with ~1000 samples.
A deeper network would overfit immediately; a wider network adds no benefit given the information content.
The model factory injects regularization into the architecture (Dropout, BatchNorm) or returns a clean
model when regularization is applied externally (L1 penalty added to loss, L2 via weight_decay).

### Cosine Annealing LR Scheduler
All experiments share a `CosineAnnealingLR` scheduler so that no optimizer is handicapped by a
poorly chosen fixed learning rate — comparisons reflect intrinsic optimizer behaviour.

In [ ]:
# ─────────────────────────────────────────────
#  Model Factory
# ─────────────────────────────────────────────

class MLP(nn.Module):
    """
    Configurable two-hidden-layer MLP for binary classification.
    Architecture adapts based on reg_config['type']:
      - none / l1 / l2  : plain linear + ReLU stack
      - dropout          : Dropout after each hidden activation
      - label_smooth     : same as none (smoothing applied in loss)
      - bn_dropout       : BatchNorm before each linear + Dropout after activation
    """
    def __init__(self, input_dim, hidden1, hidden2, reg_config):
        super().__init__()
        t      = reg_config.get('type', 'none')
        p      = reg_config.get('rate',  0.0)
        use_bn = (t == 'bn_dropout')
        use_dp = (t in ('dropout', 'bn_dropout'))
        layers = []
        if use_bn:
            layers.append(nn.BatchNorm1d(input_dim))
        layers.append(nn.Linear(input_dim, hidden1))
        layers.append(nn.ReLU())
        if use_dp:
            layers.append(nn.Dropout(p=p))
        if use_bn:
            layers.append(nn.BatchNorm1d(hidden1))
        layers.append(nn.Linear(hidden1, hidden2))
        layers.append(nn.ReLU())
        if use_dp:
            layers.append(nn.Dropout(p=p * 0.5))
        layers.append(nn.Linear(hidden2, 1))
        self.net = nn.Sequential(*layers)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x).squeeze(-1)


# ─────────────────────────────────────────────
#  Optimizer Factory
# ─────────────────────────────────────────────

def build_optimizer(model, opt_cfg, reg_cfg):
    """Weight decay uses explicit None-check so weight_decay=0.0 is respected."""
    wd_reg = reg_cfg.get('weight_decay')
    wd_opt = opt_cfg.get('weight_decay')
    wd     = wd_reg if wd_reg is not None else (wd_opt if wd_opt is not None else 0.0)
    lr     = opt_cfg['lr']
    t      = opt_cfg['type']
    if t == 'sgd':
        return torch.optim.SGD(
            model.parameters(), lr=lr,
            momentum=opt_cfg.get('momentum', 0.0),
            weight_decay=wd,
        )
    if t == 'adam':
        return torch.optim.Adam(model.parameters(),   lr=lr, weight_decay=wd)
    if t == 'adamw':
        return torch.optim.AdamW(model.parameters(),  lr=lr, weight_decay=wd)
    if t == 'rmsprop':
        return torch.optim.RMSprop(model.parameters(), lr=lr, weight_decay=wd)
    raise ValueError(f'Unknown optimizer type: {t!r}')


# ─────────────────────────────────────────────
#  Label Smoothing Loss
# ─────────────────────────────────────────────

class LabelSmoothingBCE(nn.Module):
    """
    Binary cross-entropy with label smoothing.
    Softens hard 0/1 targets to (s/2, 1-s/2), reducing overconfident logits.
    Particularly useful in small medical datasets where label noise is high.
    """
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing

    def forward(self, logits, targets):
        targets_s = targets * (1.0 - self.smoothing) + 0.5 * self.smoothing
        return F.binary_cross_entropy_with_logits(logits, targets_s)


def get_criterion(reg_cfg):
    if reg_cfg.get('type') == 'label_smooth':
        return LabelSmoothingBCE(smoothing=reg_cfg.get('smoothing', 0.1))
    return nn.BCEWithLogitsLoss()


def l1_penalty(model, lambda_):
    return lambda_ * sum(p.abs().sum() for p in model.parameters() if p.requires_grad)


print('Model factory defined.')

In [ ]:
# ─────────────────────────────────────────────
#  Training & Evaluation Loops
# ─────────────────────────────────────────────

def train_epoch(model, loader, optimizer, criterion, reg_cfg, device):
    model.train()
    total_loss = 0.0
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_b), y_b)
        if reg_cfg.get('type') == 'l1':
            loss = loss + l1_penalty(model, reg_cfg['l1_lambda'])
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(X_b)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def eval_loader(model, loader, criterion, reg_cfg, device):
    """Reports pure criterion loss (no L1 term) so losses are comparable across regularizers."""
    model.eval()
    total_loss, all_probs, all_targets = 0.0, [], []
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        logits = model(X_b)
        loss   = criterion(logits, y_b)
        total_loss += loss.item() * len(X_b)
        all_probs.extend(torch.sigmoid(logits).cpu().numpy())
        all_targets.extend(y_b.cpu().numpy())
    probs   = np.array(all_probs)
    targets = np.array(all_targets)
    preds   = (probs >= 0.5).astype(int)
    return {
        'loss':     total_loss / len(loader.dataset),
        'accuracy': float(accuracy_score(targets, preds)),
        'f1':       float(f1_score(targets, preds, zero_division=0)),
        'roc_auc':  float(roc_auc_score(targets, probs)),
    }


def run_experiment(reg_cfg, opt_cfg, device, epochs=EPOCHS, patience=EARLY_STOP_PAT):
    torch.manual_seed(SEED)
    model     = MLP(INPUT_DIM, HIDDEN1, HIDDEN2, reg_cfg).to(device)
    optimizer = build_optimizer(model, opt_cfg, reg_cfg)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = get_criterion(reg_cfg)

    history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': [], 'val_roc_auc': []}
    best_val_loss = float('inf')
    best_state    = None
    no_improve    = 0

    for epoch in range(epochs):
        tr_loss = train_epoch(model, train_loader, optimizer, criterion, reg_cfg, device)
        val_m   = eval_loader(model, val_loader,   criterion, reg_cfg, device)
        scheduler.step()
        history['train_loss'].append(tr_loss)
        history['val_loss'].append(val_m['loss'])
        history['val_acc'].append(val_m['accuracy'])
        history['val_f1'].append(val_m['f1'])
        history['val_roc_auc'].append(val_m['roc_auc'])
        if val_m['loss'] < best_val_loss:
            best_val_loss = val_m['loss']
            best_state    = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve    = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    model.load_state_dict(best_state)
    test_m = eval_loader(model, test_loader, criterion, reg_cfg, device)
    return {
        'history':       history,
        'test_metrics':  test_m,
        'stopped_epoch': len(history['train_loss']),
        'model':         model,
    }


# ─────────────────────────────────────────────
#  Plotly table helper (no matplotlib dependency)
# ─────────────────────────────────────────────

def make_table_fig(df, highlight_cols=(), title=''):
    """Plotly table with YlGn gradient on highlight_cols (higher = greener)."""
    import plotly.colors as _pc
    df = df.copy().reset_index(drop=True)
    n  = len(df)
    cell_colors = []
    for col in df.columns:
        if col in highlight_cols:
            vals  = df[col].astype(float).values
            vmin, vmax = vals.min(), vals.max()
            denom = vmax - vmin if vmax > vmin else 1.0
            norms = (vals - vmin) / denom
            cell_colors.append([_pc.sample_colorscale('YlGn', float(v))[0] for v in norms])
        else:
            cell_colors.append(['white'] * n)
    cell_vals = []
    for col in df.columns:
        if pd.api.types.is_float_dtype(df[col]):
            cell_vals.append([f'{v:.4f}' for v in df[col]])
        else:
            cell_vals.append(df[col].tolist())
    fig = go.Figure(data=[go.Table(
        header=dict(
            values=[f'<b>{c}</b>' for c in df.columns],
            fill_color='#2c5f8a', font=dict(color='white', size=12), align='center',
        ),
        cells=dict(
            values=cell_vals, fill_color=cell_colors,
            align='center', font=dict(size=11), height=28,
        ),
    )])
    fig.update_layout(
        title=title, template='plotly_white',
        margin=dict(l=0, r=0, t=40 if title else 10, b=0),
        height=max(150, 50 + n * 32),
    )
    return fig


print('Training engine ready.')

## 6. Phase 1 — Regularizer Ablation

All regularizers from `REGULARIZERS` are trained with the same fixed optimizer (`PHASE1_OPTIMIZER_NAME`).
This isolates the effect of regularization, holding optimizer behaviour constant.

In [ ]:
phase1_opt_cfg = next(o for o in OPTIMIZERS if o['name'] == PHASE1_OPTIMIZER_NAME)
phase1_raw     = []

print(f'Phase 1: all regularizers  ×  optimizer = {PHASE1_OPTIMIZER_NAME}')
print('=' * 65)

for reg in REGULARIZERS:
    print(f'  {reg["name"]:<22}', end=' ', flush=True)
    result = run_experiment(reg, phase1_opt_cfg, DEVICE)
    phase1_raw.append({
        'phase':       'Phase 1',
        'regularizer': reg['name'],
        'optimizer':   PHASE1_OPTIMIZER_NAME,
        **{f'test_{k}': v for k, v in result['test_metrics'].items()},
        'stopped_epoch': result['stopped_epoch'],
        '_history':    result['history'],
        '_model':      result['model'],
    })
    m = result['test_metrics']
    print(f'  AUC={m["roc_auc"]:.4f}  F1={m["f1"]:.4f}  Acc={m["accuracy"]:.4f}  ep={result["stopped_epoch"]}')

print('\nPhase 1 complete.')

### Phase 1 Results — Visualizations

In [ ]:
df_p1 = pd.DataFrame(
    [{k: v for k, v in r.items() if not k.startswith('_')} for r in phase1_raw]
)

# ── Grouped bar: all metrics by regularizer ─────────────────────────
df_melt = df_p1.melt(
    id_vars='regularizer',
    value_vars=list(METRIC_MAP.keys()),
    var_name='metric', value_name='score',
)
df_melt['metric'] = df_melt['metric'].map(METRIC_MAP)

fig_p1_bar = px.bar(
    df_melt, x='regularizer', y='score', color='metric',
    barmode='group',
    title=f'Phase 1: Regularizer Comparison (optimizer = {PHASE1_OPTIMIZER_NAME})',
    labels={'score': 'Score', 'regularizer': 'Regularizer'},
    color_discrete_sequence=px.colors.qualitative.Plotly,
    template='plotly_white', text_auto='.3f',
)
fig_p1_bar.update_layout(yaxis=dict(range=[0.4, 1.05]))
fig_p1_bar.update_traces(textposition='outside')
fig_p1_bar.show()

# ── Stopped-epoch bar ────────────────────────────────────────
px.bar(
    df_p1, x='regularizer', y='stopped_epoch',
    title='Phase 1: Epochs Until Early Stopping',
    color='regularizer', template='plotly_white', text_auto=True,
).update_traces(textposition='outside').show()

In [ ]:
# ── Learning curves + overfitting gap ────────────────────────
# 3rd panel (val − train gap) is the key regularization signal:
# a large/growing gap = overfitting; a small gap = good generalization.
fig_lc = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Train Loss', 'Validation Loss', 'Overfitting Gap (val − train)'),
)
colors_p1 = px.colors.qualitative.Plotly
for idx, r in enumerate(phase1_raw):
    hist  = r['_history']
    ep    = list(range(1, len(hist['train_loss']) + 1))
    color = colors_p1[idx % len(colors_p1)]
    name  = r['regularizer']
    gap   = [v - t for v, t in zip(hist['val_loss'], hist['train_loss'])]

    fig_lc.add_trace(go.Scatter(x=ep, y=hist['train_loss'], name=name,
                                line=dict(color=color), legendgroup=name),
                     row=1, col=1)
    fig_lc.add_trace(go.Scatter(x=ep, y=hist['val_loss'], name=name,
                                line=dict(color=color, dash='dot'),
                                legendgroup=name, showlegend=False),
                     row=1, col=2)
    fig_lc.add_trace(go.Scatter(x=ep, y=gap, name=name,
                                line=dict(color=color, dash='dashdot'),
                                legendgroup=name, showlegend=False),
                     row=1, col=3)

fig_lc.add_hline(y=0, line_dash='dash', line_color='grey',
                 annotation_text='no gap', row=1, col=3)
fig_lc.update_layout(title='Phase 1: Learning Curves by Regularizer',
                     template='plotly_white', height=420)
fig_lc.show()

# ── Val ROC-AUC over epochs ───────────────────────────────────
fig_auc = go.Figure()
for idx, r in enumerate(phase1_raw):
    hist = r['_history']
    ep   = list(range(1, len(hist['val_roc_auc']) + 1))
    fig_auc.add_trace(go.Scatter(x=ep, y=hist['val_roc_auc'],
                                  name=r['regularizer'],
                                  line=dict(color=colors_p1[idx % len(colors_p1)])))
fig_auc.update_layout(title='Phase 1: Validation ROC-AUC over Epochs',
                       xaxis_title='Epoch', yaxis_title='ROC-AUC',
                       template='plotly_white')
fig_auc.show()

## 7. Automatic Best-Regularizer Selection

The best regularizer (by ROC-AUC on the test set) is identified automatically.  
Phase 2 uses it as the fixed regularizer — no manual intervention required.

In [ ]:
best_p1_idx   = df_p1['test_roc_auc'].idxmax()
best_p1_row   = df_p1.loc[best_p1_idx]
BEST_REG_NAME = best_p1_row['regularizer']
best_reg_cfg  = next(r for r in REGULARIZERS if r['name'] == BEST_REG_NAME)

print(f'  Best regularizer : {BEST_REG_NAME}')
print(f'  ROC-AUC          : {best_p1_row["test_roc_auc"]:.4f}')
print(f'  F1-Score         : {best_p1_row["test_f1"]:.4f}')
print(f'  Accuracy         : {best_p1_row["test_accuracy"]:.4f}')
print(f'\n=> Phase 2 will use \'{BEST_REG_NAME}\' as fixed regularizer.')

make_table_fig(
    df_p1.sort_values('test_roc_auc', ascending=False),
    highlight_cols=('test_roc_auc', 'test_f1', 'test_accuracy'),
    title='Phase 1: Regularizer Results (sorted by ROC-AUC)',
).show()

## 8. Phase 2 — Optimizer Ablation

All optimizers from `OPTIMIZERS` are trained with the best regularizer found in Phase 1.
This isolates the effect of the optimizer, holding regularization behaviour constant.

In [ ]:
phase2_raw = []

print(f'Phase 2: all optimizers  ×  regularizer = {BEST_REG_NAME}')
print('=' * 65)

for opt in OPTIMIZERS:
    print(f'  {opt["name"]:<18}', end=' ', flush=True)
    result = run_experiment(best_reg_cfg, opt, DEVICE)
    phase2_raw.append({
        'phase':       'Phase 2',
        'regularizer': BEST_REG_NAME,
        'optimizer':   opt['name'],
        **{f'test_{k}': v for k, v in result['test_metrics'].items()},
        'stopped_epoch': result['stopped_epoch'],
        '_history':    result['history'],
        '_model':      result['model'],
    })
    m = result['test_metrics']
    print(f'  AUC={m["roc_auc"]:.4f}  F1={m["f1"]:.4f}  Acc={m["accuracy"]:.4f}  ep={result["stopped_epoch"]}')

print('\nPhase 2 complete.')

### Phase 2 Results — Visualizations

In [ ]:
df_p2 = pd.DataFrame(
    [{k: v for k, v in r.items() if not k.startswith('_')} for r in phase2_raw]
)

df_melt2 = df_p2.melt(
    id_vars='optimizer',
    value_vars=list(METRIC_MAP.keys()),
    var_name='metric', value_name='score',
)
df_melt2['metric'] = df_melt2['metric'].map(METRIC_MAP)

fig_p2_bar = px.bar(
    df_melt2, x='optimizer', y='score', color='metric',
    barmode='group',
    title=f'Phase 2: Optimizer Comparison (regularizer = {BEST_REG_NAME})',
    labels={'score': 'Score', 'optimizer': 'Optimizer'},
    color_discrete_sequence=px.colors.qualitative.Plotly,
    template='plotly_white', text_auto='.3f',
)
fig_p2_bar.update_layout(yaxis=dict(range=[0.4, 1.05]))
fig_p2_bar.update_traces(textposition='outside')
fig_p2_bar.show()

# ── Learning curves + overfitting gap ────────────────────────
fig_lc2 = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Train Loss', 'Validation Loss', 'Overfitting Gap (val − train)'),
)
colors_p2 = px.colors.qualitative.Safe
for idx, r in enumerate(phase2_raw):
    hist  = r['_history']
    ep    = list(range(1, len(hist['train_loss']) + 1))
    color = colors_p2[idx % len(colors_p2)]
    name  = r['optimizer']
    gap   = [v - t for v, t in zip(hist['val_loss'], hist['train_loss'])]

    fig_lc2.add_trace(go.Scatter(x=ep, y=hist['train_loss'], name=name,
                                  line=dict(color=color), legendgroup=name), row=1, col=1)
    fig_lc2.add_trace(go.Scatter(x=ep, y=hist['val_loss'], name=name,
                                  line=dict(color=color, dash='dot'),
                                  legendgroup=name, showlegend=False), row=1, col=2)
    fig_lc2.add_trace(go.Scatter(x=ep, y=gap, name=name,
                                  line=dict(color=color, dash='dashdot'),
                                  legendgroup=name, showlegend=False), row=1, col=3)

fig_lc2.add_hline(y=0, line_dash='dash', line_color='grey',
                   annotation_text='no gap', row=1, col=3)
fig_lc2.update_layout(title='Phase 2: Learning Curves by Optimizer',
                       template='plotly_white', height=420)
fig_lc2.show()

# ── Val ROC-AUC over epochs ───────────────────────────────────
fig_auc2 = go.Figure()
for idx, r in enumerate(phase2_raw):
    hist = r['_history']
    ep   = list(range(1, len(hist['val_roc_auc']) + 1))
    fig_auc2.add_trace(go.Scatter(x=ep, y=hist['val_roc_auc'],
                                   name=r['optimizer'],
                                   line=dict(color=colors_p2[idx % len(colors_p2)])))
fig_auc2.update_layout(title='Phase 2: Validation ROC-AUC over Epochs',
                        xaxis_title='Epoch', yaxis_title='ROC-AUC',
                        template='plotly_white')
fig_auc2.show()

## 9. Combined Results & Cross-Phase Analysis

In [ ]:
df_all = pd.DataFrame(
    [{k: v for k, v in r.items() if not k.startswith('_')}
     for r in phase1_raw + phase2_raw]
).sort_values('test_roc_auc', ascending=False).reset_index(drop=True)

make_table_fig(
    df_all,
    highlight_cols=('test_roc_auc', 'test_f1', 'test_accuracy'),
    title='All Experiments (sorted by ROC-AUC)',
).show()

In [ ]:
# ── Parallel coordinates ────────────────────────────────────
df_pc = df_all.copy()
reg_cats = {v: i for i, v in enumerate(df_pc['regularizer'].unique())}
opt_cats = {v: i for i, v in enumerate(df_pc['optimizer'].unique())}
df_pc['reg_idx'] = df_pc['regularizer'].map(reg_cats)
df_pc['opt_idx'] = df_pc['optimizer'].map(opt_cats)

go.Figure(go.Parcoords(
    line=dict(color=df_pc['test_roc_auc'], colorscale='Viridis', showscale=True,
              colorbar=dict(title='ROC-AUC')),
    dimensions=[
        dict(label='Regularizer', values=df_pc['reg_idx'],
             tickvals=list(reg_cats.values()), ticktext=list(reg_cats.keys())),
        dict(label='Optimizer',   values=df_pc['opt_idx'],
             tickvals=list(opt_cats.values()), ticktext=list(opt_cats.keys())),
        dict(label='ROC-AUC',      values=df_pc['test_roc_auc']),
        dict(label='F1-Score',     values=df_pc['test_f1']),
        dict(label='Accuracy',     values=df_pc['test_accuracy']),
        dict(label='Stopped Epoch',values=df_pc['stopped_epoch']),
    ],
)).update_layout(title='Parallel Coordinates: All Experiments (brush to filter)',
                 template='plotly_white', height=500).show()

# ── ROC-AUC ranking bar ──────────────────────────────────
df_rank = df_all.copy()
df_rank['run'] = df_rank['regularizer'] + ' / ' + df_rank['optimizer']
px.bar(
    df_rank.sort_values('test_roc_auc'),
    x='test_roc_auc', y='run', color='phase',
    orientation='h', title='All Runs Ranked by ROC-AUC',
    labels={'test_roc_auc': 'ROC-AUC', 'run': 'Regularizer / Optimizer'},
    template='plotly_white', text_auto='.4f',
).update_layout(xaxis=dict(range=[0.5, 1.0])).show()

## 10. Best Configuration Selection

In [ ]:
best_row       = df_all.iloc[0]
FINAL_REG_NAME = best_row['regularizer']
FINAL_OPT_NAME = best_row['optimizer']
final_reg_cfg  = next(r for r in REGULARIZERS if r['name'] == FINAL_REG_NAME)
final_opt_cfg  = next(o for o in OPTIMIZERS   if o['name'] == FINAL_OPT_NAME)

print('Best configuration found:')
print(f'  Regularizer : {FINAL_REG_NAME}')
print(f'  Optimizer   : {FINAL_OPT_NAME}')
print(f'  ROC-AUC     : {best_row["test_roc_auc"]:.4f}')
print(f'  F1-Score    : {best_row["test_f1"]:.4f}')
print(f'  Accuracy    : {best_row["test_accuracy"]:.4f}')

## 11. Final Model — Stochastic Weight Averaging (SWA)

**Why SWA?**  
Standard SGD/Adam converge to sharp minima that generalise poorly. SWA averages the weights
of multiple snapshots taken near the end of training, landing in a *flat* region of the loss
surface — flat minima are known to generalise better (Izmailov et al., 2018).

SWA requires no additional gradient steps: it reuses the existing training loop, averaging
weights in the final 25% of epochs at the cost of a single `update_bn` pass.
This qualifies as a **creative, beyond-syllabus** regularisation strategy.

In [ ]:
SWA_EPOCHS   = EPOCHS
SWA_LR       = 5e-4
SWA_START_EP = int(SWA_EPOCHS * SWA_START_FRAC)

torch.manual_seed(SEED)
model_base = MLP(INPUT_DIM, HIDDEN1, HIDDEN2, final_reg_cfg).to(DEVICE)
opt_base   = build_optimizer(model_base, final_opt_cfg, final_reg_cfg)
swa_model  = AveragedModel(model_base)
swa_sched  = SWALR(opt_base, swa_lr=SWA_LR, anneal_epochs=10)
criterion  = get_criterion(final_reg_cfg)
cos_sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt_base, T_max=SWA_START_EP)

swa_history = {'train_loss': [], 'val_loss': []}

print(f'SWA retraining  -  base optimizer: {FINAL_OPT_NAME}  |  regularizer: {FINAL_REG_NAME}')
print(f'Cosine annealing epochs 1–{SWA_START_EP}, SWA averaging from {SWA_START_EP + 1}–{SWA_EPOCHS}')

for epoch in range(1, SWA_EPOCHS + 1):
    tr_loss = train_epoch(model_base, train_loader, opt_base, criterion, final_reg_cfg, DEVICE)
    val_m   = eval_loader(model_base, val_loader,   criterion, final_reg_cfg, DEVICE)
    swa_history['train_loss'].append(tr_loss)
    swa_history['val_loss'].append(val_m['loss'])
    if epoch <= SWA_START_EP:
        cos_sched.step()
    else:
        swa_model.update_parameters(model_base)
        swa_sched.step()

update_bn(train_loader, swa_model, device=DEVICE)
print('\nSWA training complete. BatchNorm stats updated.')

### Final Model Evaluation

In [ ]:
swa_test  = eval_loader(swa_model,  test_loader, criterion, final_reg_cfg, DEVICE)
base_test = eval_loader(model_base, test_loader, criterion, final_reg_cfg, DEVICE)

comparison = pd.DataFrame([
    {'model': f'Base ({FINAL_REG_NAME} / {FINAL_OPT_NAME})', **base_test},
    {'model': f'SWA  ({FINAL_REG_NAME} / {FINAL_OPT_NAME})', **swa_test},
    {'model': 'Best ablation run',
     'loss':     best_row['test_loss'],
     'accuracy': best_row['test_accuracy'],
     'f1':       best_row['test_f1'],
     'roc_auc':  best_row['test_roc_auc']},
])

print('Final comparison:')
make_table_fig(
    comparison,
    highlight_cols=('roc_auc', 'f1', 'accuracy'),
    title='Final Model Comparison',
).show()

# ── SWA training curve ────────────────────────────────────
ep = list(range(1, SWA_EPOCHS + 1))
fig_swa = go.Figure()
fig_swa.add_trace(go.Scatter(x=ep, y=swa_history['train_loss'],
                              name='Train Loss', line=dict(color='steelblue')))
fig_swa.add_trace(go.Scatter(x=ep, y=swa_history['val_loss'],
                              name='Val Loss', line=dict(color='coral', dash='dot')))
fig_swa.add_vline(x=SWA_START_EP, line_dash='dash', line_color='green',
                   annotation_text='SWA averaging starts',
                   annotation_position='top right')
fig_swa.update_layout(
    title=f'Final Model (SWA): Training Curve - {FINAL_REG_NAME} / {FINAL_OPT_NAME}',
    xaxis_title='Epoch', yaxis_title='Loss', template='plotly_white',
)
fig_swa.show()

In [ ]:
from sklearn.metrics import roc_curve, confusion_matrix, precision_recall_curve, average_precision_score

swa_probs, swa_targets = get_probs_targets(swa_model, test_loader, DEVICE)
swa_preds = (swa_probs >= 0.5).astype(int)

# ── ROC curve ────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(swa_targets, swa_probs)
auc_val     = roc_auc_score(swa_targets, swa_probs)

fig_roc = go.Figure()
fig_roc.add_trace(go.Scatter(
    x=fpr, y=tpr,
    fill='tozeroy', fillcolor='rgba(99,110,250,0.15)',
    line=dict(color='royalblue', width=2),
    name=f'SWA Model (AUC = {auc_val:.4f})',
))
fig_roc.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], line=dict(color='grey', dash='dash'),
    name='Random Classifier',
))
fig_roc.update_layout(
    title='Final SWA Model - ROC Curve (Test Set)',
    xaxis_title='False Positive Rate', yaxis_title='True Positive Rate',
    template='plotly_white', legend=dict(x=0.6, y=0.1),
)
fig_roc.show()

# ── Precision-Recall curve ────────────────────────────────────
# More informative than ROC for medical datasets:
# recall = sensitivity (catching disease), precision = positive predictive value.
precision_vals, recall_vals, _ = precision_recall_curve(swa_targets, swa_probs)
ap = average_precision_score(swa_targets, swa_probs)
baseline_precision = swa_targets.mean()  # no-skill baseline = prevalence

fig_pr = go.Figure()
fig_pr.add_trace(go.Scatter(
    x=recall_vals, y=precision_vals,
    fill='tozeroy', fillcolor='rgba(0,204,150,0.12)',
    line=dict(color='seagreen', width=2),
    name=f'SWA Model (AP = {ap:.4f})',
))
fig_pr.add_hline(y=baseline_precision, line_dash='dash', line_color='grey',
                  annotation_text=f'No-skill baseline ({baseline_precision:.2f})',
                  annotation_position='top right')
fig_pr.update_layout(
    title='Final SWA Model - Precision-Recall Curve (Test Set)',
    xaxis_title='Recall (Sensitivity)', yaxis_title='Precision (PPV)',
    template='plotly_white', legend=dict(x=0.05, y=0.05),
    yaxis=dict(range=[0, 1.05]),
)
fig_pr.show()

# ── Confusion matrix ─────────────────────────────────────────
# For medical classification: FN (missed disease) >> FP (false alarm).
cm = confusion_matrix(swa_targets, swa_preds)
labels = ['No Disease', 'Has Disease']

fig_cm = px.imshow(
    cm,
    text_auto=True,
    x=labels, y=labels,
    color_continuous_scale='Blues',
    title='Confusion Matrix - SWA Model (Test Set)',
    labels=dict(x='Predicted', y='Actual', color='Count'),
    template='plotly_white',
    aspect='equal',
)
fig_cm.update_layout(
    xaxis_title='Predicted Label',
    yaxis_title='True Label',
    coloraxis_showscale=False,
)

tn, fp, fn, tp = cm.ravel()
print(f'Confusion Matrix:')
print(f'  True Positives  (TP): {tp}  - correctly identified disease')
print(f'  True Negatives  (TN): {tn}  - correctly identified no disease')
print(f'  False Positives (FP): {fp}  - false alarm (no disease, predicted disease)')
print(f'  False Negatives (FN): {fn}  - missed disease (disease, predicted no disease)')
print(f'  Sensitivity (Recall): {tp/(tp+fn):.4f}')
print(f'  Specificity:          {tn/(tn+fp):.4f}')
fig_cm.show()

## 12. Conclusions

### Two-phase ablation design — rationale
A full Cartesian product (N × M runs) would conflate regularizer and optimizer effects, making
individual contributions harder to isolate. The two-phase approach mirrors a controlled experiment:
one variable changes at a time, and the winner of each phase informs the next. This is both
more interpretable and computationally frugal.

### SWA — beyond the syllabus
Stochastic Weight Averaging (Izmailov et al., 2018) was selected as the final fine-tuning strategy
because it requires no new hyperparameter search: it reuses the existing training loop, averaging
weights in the final 25% of epochs. It is particularly effective on small datasets where sharp
minima are most likely to appear.

### Possible improvements
- Bayesian hyperparameter optimisation (Optuna) for LR and architecture search
- Feature selection via permutation importance
- Ensemble of the top-3 ablation models
- Cross-validation for more robust metric estimates on 1025 samples